## Transform US Census GeoSpatial data on Hugging Face to parquet using DuckDB



In [ ]:
import duckdb
import os
from google.colab import userdata


In [ ]:
# Connect to DuckDB and load the spatial extension
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;") ;


In [ ]:
roads_url = "https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/tl_2024_35006_roads.zip"
states_url = "https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/tl_2024_us_state.zip"


In [ ]:
# Extract the base file name
roads_base = roads_url.split("/")[-1].split(".")[0]
states_base = states_url.split("/")[-1].split(".")[0]

roads_base, states_base


In [ ]:
! curl -L -O {roads_url}
! curl -L -O {states_url}


In [ ]:
! ls -la {roads_base}* {states_base}*


In [ ]:
roads_path = f"/vsizip/{roads_base}.zip"
states_path = f"/vsizip/{states_base}.zip"

roads_path, states_path


## Convert to Parquet


In [ ]:
# Export Roads shapefile to GeoParquet
con.execute(f"""
    COPY (
        SELECT * FROM st_read('{roads_path}')
    ) TO '{roads_base}.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
""")



In [ ]:
# Export States shapefile to GeoParquet
con.execute(f"""
    COPY (
        SELECT * FROM st_read("{states_path}")
    ) TO '{states_base}.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
""")


In [ ]:
query = f"""
    SELECT
        roads.* EXCLUDE (geom),
        states.STATEFP,
        states.STUSPS,
        states.NAME AS state_name,
        roads.geom AS road_geometry,
        states.geom AS state_geometry
    FROM read_parquet('{roads_base}.parquet') roads
    LEFT JOIN read_parquet('{states_base}.parquet') states
        ON ST_Within(roads.geom, states.geom)
    WHERE roads.FULLNAME LIKE '%I- 25%'
       OR roads.RTTYP = 'I'
"""
print(query)

gdf_combined = con.sql(query).df()
gdf_combined


## Push to Hugging Face


## Configure settings and authenticate to Hugging Face


In [ ]:
hf_account = os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account


In [ ]:
hf_repo = os.environ["HF_REPO"] = "DUC-2026"
hf_repo


In [ ]:
# Retrieve the token from Colab's secret manager
_ = hf_token = os.environ["HF_TOKEN"] = userdata.get('hf_token')
f"{_[:3]} ... {_[-4:]}"


In [ ]:
# Log in automatically without an interactive prompt
! hf auth login --token ${HF_TOKEN}


## Push the parquet files to Hugging Face


### State

In [ ]:
hf_file = os.environ["HF_FILE"] = f"{states_base}.parquet"
hf_file


In [ ]:
%%capture hf_upload
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

# Comment out the echo to make this live
# echo DRY-RUN: \
hf upload \
  --type dataset \
  "${TARGET_REPO}" \
  "${LOCAL_SOURCE}" \
  "${REMOTE_DESTINATION}"


In [ ]:
print(hf_upload.stdout)


### Roads

In [ ]:
hf_file = os.environ["HF_FILE"] = f"{roads_base}.parquet"
hf_file


In [ ]:
%%capture hf_upload
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

# Comment out the echo to make this live
# echo DRY-RUN: \
hf upload \
  --type dataset \
  "${TARGET_REPO}" \
  "${LOCAL_SOURCE}" \
  "${REMOTE_DESTINATION}"


In [ ]:
print(hf_upload.stdout)


## Verify


In [ ]:
roads_parquet_url = 'https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/tl_2024_35006_roads.parquet'
states_parquet_url = 'https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/tl_2024_us_state.parquet'

roads_parquet_url, states_parquet_url


In [ ]:
query = f"""
    SELECT
        roads.* EXCLUDE (geom),
        states.STATEFP,
        states.STUSPS,
        states.NAME AS state_name,
        roads.geom AS road_geometry,
        states.geom AS state_geometry
    FROM read_parquet('{roads_parquet_url}') roads
    LEFT JOIN read_parquet('{states_parquet_url}') states
        ON ST_Within(roads.geom, states.geom)
    WHERE roads.FULLNAME LIKE '%I- 25%'
       OR roads.RTTYP = 'I'
"""
print(query)

gdf_combined = con.sql(query).df()
gdf_combined
